In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
    
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata_cospar = ad.read_h5ad("./data/larry/cospar_tmap_result.h5ad")
print(adata)
print(adata_cospar)

In [ ]:
import numpy as np

def match_by_X_emb(adata, adata_cospar):
    X1 = adata.obsm["X_emb"]
    X2 = adata_cospar.obsm["X_emb"]

    # Convert rows to tuples (hashable, exact float match)
    rows1 = [tuple(r) for r in X1]
    rows2 = [tuple(r) for r in X2]

    # Build lookup: embedding -> index in adata_cospar
    lookup = {row: j for j, row in enumerate(rows2)}

    # Map each row in adata → adata_cospar index (or -1 if missing)
    mapA2B = np.array([lookup.get(row, -1) for row in rows1], dtype=int)

    return mapA2B

mapA2B = match_by_X_emb(adata, adata_cospar)
print("Matched:", np.sum(mapA2B != -1), "/", len(mapA2B))

In [ ]:
from scipy.spatial import cKDTree
import joblib

# ===============================================================
# 0. Load embedder
# ===============================================================
emb = joblib.load("./data/larry/larry_flowmap_embedder.pkl")
emb.cospar_index = mapA2B     # your mapping (adata → adata_cospar)


# ===============================================================
# 1. Tentatively assign new_X_emb from FlowMap
# ===============================================================
n_cospar = adata_cospar.n_obs
new_X_emb = np.full((n_cospar, 2), np.nan)   # initialize with NaNs

idx = emb.cospar_index
mask = idx != -1                             # valid matches only

# fill matched rows
new_X_emb[idx[mask]] = emb.X_emb[mask]


# ===============================================================
# 2. Identify rows with NaN in new_X_emb
# ===============================================================
nan_rows = np.where(~np.isfinite(new_X_emb).all(axis=1))[0]
print("Cells needing patching:", len(nan_rows))


# ===============================================================
# 3. Find nearest neighbors in *original CoSpar embedding*
# ===============================================================
X_cospar_orig = adata_cospar.obsm["X_emb"]    # THIS is the reference geometry

# Use only rows whose new_X_emb is NOT NaN
valid_rows = np.where(np.isfinite(new_X_emb).all(axis=1))[0]

# KD-tree in original embedding space, but only valid rows
tree = cKDTree(X_cospar_orig[valid_rows])

# query NN for each NaN row
_, nn_idx = tree.query(X_cospar_orig[nan_rows], k=1)

# convert relative NN indices → global indices
global_nn_idx = valid_rows[nn_idx]


# ===============================================================
# 4. Replace NaN rows in new_X_emb with the NN's embedding
# ===============================================================
new_X_emb[nan_rows] = new_X_emb[global_nn_idx]


# ===============================================================
# 5. Store the patched embedding back into CoSpar object
# ===============================================================
adata_cospar.obsm["X_emb"] = new_X_emb

print("DONE: adata_cospar.X_emb refreshed and patched.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from flowmap.geometry.curvature import compute_flow_curvature

curv = compute_flow_curvature(emb)

In [ ]:
X = emb.X_emb

k_total  = curv["curvature"]["total"]
k_geod   = curv["curvature"]["geodesic"]
k_normal = curv["curvature"]["normal"]

def clip_quantile(arr, q_low=2, q_high=98):
    lo, hi = np.percentile(arr, [q_low, q_high])
    return np.clip(arr, lo, hi)

k_total_c  = clip_quantile(k_total)
k_geod_c   = clip_quantile(k_geod)
k_normal_c = clip_quantile(k_normal)

fig, axes = plt.subplots(1,3, figsize=(18,5))

axes[0].scatter(X[:,0], X[:,1], c=k_total_c, s=6, cmap="coolwarm")
axes[0].set_title("Total curvature")

axes[1].scatter(X[:,0], X[:,1], c=k_geod_c, s=6, cmap="coolwarm")
axes[1].set_title("Geodesic curvature")

axes[2].scatter(X[:,0], X[:,1], c=k_normal_c, s=6, cmap="coolwarm")
axes[2].set_title("Normal curvature")

for ax in axes:
    ax.set_aspect("equal")
    ax.axis("off")

plt.show()